In [ ]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch
import numpy as np

# IEEE/Elsevier Double-Column Formatting Standard
plt.rcParams.update({
    'figure.dpi': 600,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 7,
    'axes.labelsize': 7,
    'axes.titlesize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 6,
    'figure.titlesize': 8,
    'lines.linewidth': 1.0,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.4
})

# Anchor to the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from src.config import SimConfig, EnvConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker, print_markdown_table
from src.utils.plotting import plot_simulation_dashboard, plot_simulation_dashboard_bis
from src.solvers import AugmentedHybridSDPSolver
from src.plants import AugmentedHybridPlant
from src.controllers import build_approach, AugmentedFCLockedControl, AugmentedPolicyControl, AugmentedValueControl

In [ ]:
# 1. Setup the Environment and Lock the High-Fidelity Physics
env = EnvConfig()

# Highest acceptable fidelity for the baseline showdown
config = SimConfig(
    delta_P=100.0, 
    delta_t=300,                  
    N_d=6,  # Capped at 5 to ensure dense Markov matrices 
    delta_n=1, # 12 Individual modules (Max Fidelity)
    use_smart_grid=True,
    apply_terminal_soc_cost=True,
    alpha_fc=4
)
fleet_data = load_and_cache_entire_fleet(env)
exclude_days = [1, 2, 3]
benchmarker = VoyageBenchmarker(fleet_data, env, config, exclude_days)

In [ ]:
# 2. Define the Control Architectures
approaches = {
    "Nearest-Neighbor (PLC)": build_approach(
        controller_cls=AugmentedFCLockedControl, 
        plant_cls=AugmentedHybridPlant, 
        solver_cls=AugmentedHybridSDPSolver, 
        is_macro=False
    ),
    "Policy Interpolation": build_approach(
        controller_cls=AugmentedPolicyControl, 
        plant_cls=AugmentedHybridPlant, 
        solver_cls=AugmentedHybridSDPSolver, 
        is_macro=False
    ),
    "Continuous Value": build_approach(
        controller_cls=AugmentedValueControl, 
        plant_cls=AugmentedHybridPlant, 
        solver_cls=AugmentedHybridSDPSolver, 
        is_macro=False
    ),
}

In [ ]:
# 3. Run the Benchmarks (LOOCV)
print("--- RUNNING ARCHITECTURE SHOWDOWN (LOOCV) ---")
reports = {}

for app_name, factory in approaches.items():
    print(f"\nEvaluating {app_name}...")
    reports[app_name] = benchmarker.run_leave_one_out(factory)

summary_rows = {name: rep.summary.loc['Average'] for name, rep in reports.items()}
master_summary = pd.DataFrame(summary_rows).T

print("\n--- MASTER PERFORMANCE SUMMARY ---")
print_markdown_table(master_summary)

In [ ]:
# 4. Generate Figure 1: Architecture Showdown Bar Chart
os.makedirs('figures', exist_ok=True)
fig, ax1 = plt.subplots(figsize=(3.5, 2.5)) # Elsevier single-column width

costs = master_summary['Total Cost [$]']
x_positions = range(len(costs))

# Unique color sequence for each approach bar
bar_colors = ['#1f77b4', '#d62728', '#2ca02c']

# Plot bars with exact outline thickness requested (linewidth=0.2)
bars = ax1.bar(
    x_positions, 
    costs.values, 
    width=0.75, 
    color=bar_colors[:len(costs)], 
    edgecolor='black', 
    linewidth=0.4
)

ax1.set_ylabel("Average Total Cost [$]")
ax1.tick_params(axis='y', labelcolor='black') 
ax1.set_ylim([costs.min() * 0.95, costs.max() * 1.01])
ax1.set_xticks(x_positions)
ax1.set_xticklabels(["Discrete Policy \n Tracking", "Bilinear Policy \nSmoothing", "Continuous Value \nLookahead"], fontweight="bold", fontsize=8)

# Force grid and ticks to sit strictly behind the bars, 
# preventing bars from spilling over the bottom axis line.
ax1.set_axisbelow(True)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('figures/controllers.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Generate Figure 2: The Physical Dashboard for Continuous Value Controller
test_day = 9
run_id = f"Day {test_day}"
winning_app = "Continuous Value"

print(f"\n--- GENERATING HERO DASHBOARD FOR DAY {test_day} ({winning_app}) ---")

df_telemetry = reports[winning_app].get_telemetry(run_id)
summary_metrics = reports[winning_app].summary.loc[run_id]

# Extract terminal cost bounds to feed the plotting function
terminal_costs = (
    summary_metrics.get('Term. Switch Cost [$]', 0.0),
    summary_metrics.get('Term. SoC Cost [$]', 0.0)
)

plot_simulation_dashboard_bis(
    df=df_telemetry, 
    config=config, 
    title=f"day{test_day-3}", 
    indiv=False,
    save_plot=True
)

In [ ]:
# 6. Generate Figure 3: Daily Cost Breakdown (Stacked Single-Column)
fig, ax = plt.subplots(figsize=(3.5, 2.5)) # Elsevier single-column width

# Extract the data for the winning approach, excluding the 'Average' row
app_data = reports[winning_app].summary.drop('Average', errors='ignore')
x = np.arange(len(app_data))

# Force labels to start from "Day 1"
new_day_labels = range(1, len(x)+1)

width = 0.75
lw = 0.1 # Thin, clean outline

# Tableau 10 Scientific Palette
c_h2 = '#1f77b4'   # Blue (Base)
c_deg = '#ff7f0e'  # Orange (High contrast vs Blue)
c_bat = '#9467bd'  # Purple (High contrast vs Orange)
c_soc = '#d62728'  # Red 
c_sw = '#2ca02c'   # Green 

# Extract data components
h2_costs = app_data['H2 Fuel Cost [$]']
bat_costs = app_data['Battery Cost [$]']
deg_costs = app_data['FC Degradation Cost [$]'] + app_data['Switching Cost [$]'] + app_data['Transient Cost [$]']
term_switch = app_data['Term. Switch Cost [$]']
term_soc = app_data['Term. SoC Cost [$]']

# Build the positive stack
ax.bar(x, h2_costs, width, label='H2 Fuel', color=c_h2, edgecolor='black', linewidth=lw, zorder=3)
ax.bar(x, deg_costs, width, bottom=h2_costs, label='FC Deg.', color=c_deg, edgecolor='black', linewidth=lw, zorder=3)
ax.bar(x, bat_costs, width, bottom=h2_costs+deg_costs, label='Battery Deg.', color=c_bat, edgecolor='black', linewidth=lw, zorder=3)

pos_bottom = h2_costs + bat_costs + deg_costs

if term_switch.sum() > 0:
    ax.bar(x, term_switch, width, bottom=pos_bottom, label='Term. Switch', color=c_sw, edgecolor='black', linewidth=lw, zorder=3)
    pos_bottom += term_switch

soc_pos = np.where(term_soc > 0, term_soc, 0)
soc_neg = np.where(term_soc < 0, term_soc, 0)

if np.any(term_soc != 0):
    ax.bar(x, soc_pos, width, bottom=pos_bottom, label='Term. SoC', color=c_soc, edgecolor='black', linewidth=lw, zorder=3)
    ax.bar(x, soc_neg, width, bottom=0, color=c_soc, edgecolor='black', linewidth=lw, zorder=3)

# Formatting
ax.axhline(0, color='black', linewidth=0.5, zorder=4, linestyle="-")
ax.set_ylabel("Daily Cost [$]")
ax.set_xlabel("Day")
ax.set_xticks(x)
ax.set_xticklabels(new_day_labels)
ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)

# Expand the y-axis top limit by 25% to act as a blank canvas for the legend
current_bottom, current_top = ax.get_ylim()
ax.set_ylim(bottom=current_bottom, top=current_top * 1.25)

# Place legend cleanly inside the newly created top space
ax.legend(loc='upper right', ncol=2, frameon=True)

plt.tight_layout()
plt.savefig('figures/daily_cost_breakdown.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# -------------------------------------------------------------------------
# 1. EXTRACT AND ROUND RAW DATA (Prevents Floating Point & Summation Errors)
# -------------------------------------------------------------------------
day_data = reports[winning_app].summary.loc[run_id]

# Micro breakdown components (rounded to exact cents)
steady = round(float(day_data['FC Degradation Cost [$]']), 2)
switch = round(float(day_data['Switching Cost [$]']), 2)
trans = round(float(day_data['Transient Cost [$]']), 2)
deg_total = round(steady + switch + trans, 2)

# Macro components
h2 = round(float(day_data['H2 Fuel Cost [$]']), 2)
bat = round(float(day_data['Battery Cost [$]']), 2)

term_switch = round(float(day_data['Term. Switch Cost [$]']), 2)
term_soc = round(float(day_data['Term. SoC Cost [$]']), 2)
term_net = round(term_switch + term_soc, 2)

macro_total = round(h2 + deg_total + bat + term_net, 2)
micro_total = deg_total

# -------------------------------------------------------------------------
# 2. PRINT EXACT COST BREAKDOWN TABLE
# -------------------------------------------------------------------------
table_data = [
    ["Macro: H2 Fuel Cost", f"${h2:,.2f}"],
    ["Macro: FC Degradation Cost (Total)", f"${deg_total:,.2f}"],
    ["  ├─ Micro: Operating (Steady)", f"${steady:,.2f}"],
    ["  ├─ Micro: Switching", f"${switch:,.2f}"],
    ["  └─ Micro: Load Change (Transient)", f"${trans:,.2f}"],
    ["Macro: Battery Degradation Cost", f"${bat:,.2f}"],
    ["Macro: Terminal SOC Net Cost", f"${term_net:,.2f}"],
    ["  ├─ Micro: Term. Switch Cost", f"${term_switch:,.2f}"],
    ["  └─ Micro: Term. SOC Cost", f"${term_soc:,.2f}"],
    ["=" * 36, "=" * 12],
    ["TOTAL CUMULATIVE COST", f"${macro_total:,.2f}"]
]

df_summary = pd.DataFrame(table_data, columns=["Cost Category", "Amount [$]"])
print("\n" + "="*50)
print("           EXACT COST BREAKDOWN SUMMARY           ")
print("="*50)
print(df_summary.to_string(index=False))
print("="*50 + "\n")


# -------------------------------------------------------------------------
# 3. GENERATE FIGURE 4: WATERFALL CHART
# -------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.54, 2), gridspec_kw={'width_ratios': [5, 4]})

bar_w = 0.6
lw = 0.4 

# Colors
c_h2 = '#1f77b4'   
c_deg = '#ff7f0e'  
c_bat = '#9467bd'  
c_term_credit = '#d62728'  
c_term = '#d62728'
c_total = '#2ca02c'

# --- AXIS 1: MACRO WATERFALL ---
macro_labels = ['H2 Fuel', 'FC Deg.', 'Battery Deg.', 'Term. SOC', 'Total']
macro_vals = [h2, deg_total, bat, term_net, macro_total]
macro_colors = [c_h2, c_deg, c_bat, c_term if term_net >= 0 else c_term_credit, c_total]

running_total = 0.0
for i, (val, color) in enumerate(zip(macro_vals, macro_colors)):
    if i == len(macro_vals) - 1: 
        ax1.bar(i, val, bottom=0, width=bar_w, color=color, edgecolor='black', linewidth=lw, zorder=3)
        ax1.plot([i-1 - bar_w/2, i + bar_w/2], [running_total, val], color='black', linestyle='-', linewidth=lw, zorder=2)
        y_text = val
    else:
        bottom = running_total
        height = val
        
        if i > 0:
            ax1.plot([i-1 - bar_w/2, i + bar_w/2], [running_total, running_total], color='black', linestyle='-', linewidth=lw, zorder=2)
        
        if i == 3 and val < 0:
            bottom = running_total
            height = val 
            y_text = running_total + val
        else:
            y_text = running_total + val
            
        ax1.bar(i, height, bottom=bottom, width=bar_w, color=color, edgecolor='black', linewidth=lw, zorder=3)
        running_total += val

    offset_pts = 2 if val >= 0 else -2
    va_align = 'bottom' if val >= 0 else 'top'
    label_str = f"${val:,.0f}" if val >= 0 else f"-${abs(val):,.0f}"
    ax1.annotate(label_str, xy=(i, y_text), xytext=(0, offset_pts), textcoords="offset points", 
                 ha='center', va=va_align, fontsize=6.5, fontweight='bold', zorder=5)

ax1.set_ylabel("Cumulative Cost [$]")
ax1.set_xticks(range(len(macro_labels)))
ax1.set_xticklabels(macro_labels, fontweight="bold") 
ax1.grid(axis='y', linestyle='--', zorder=0)
ax1.text(0.02, 0.92, "(a)", transform=ax1.transAxes, fontweight='bold', fontsize=9)
ax1.set_ylim(top=ax1.get_ylim()[1] * 1.15)

# --- AXIS 2: MICRO DEGRADATION (ZOOM) ---
micro_labels = ['Operating', 'Switch', 'Load Change', 'FC Deg.']
micro_vals = [steady, switch, trans, micro_total]

micro_colors = ['#ffedd8', '#fdd0a2', '#fdae6b', c_deg]
run_micro = 0.0
for i, val in enumerate(micro_vals):
    if i == len(micro_vals) - 1:
        ax2.bar(i, val, bottom=0, width=bar_w, color=micro_colors[i], edgecolor='black', linewidth=lw, zorder=3)
        ax2.plot([i-1 - bar_w/2, i + bar_w/2], [run_micro, val], color='black', linestyle='-', linewidth=lw, zorder=2)
        y_text = val
    else:
        if i > 0:
            ax2.plot([i-1 - bar_w/2, i + bar_w/2], [run_micro, run_micro], color='black', linestyle='-', linewidth=lw, zorder=2)
        ax2.bar(i, val, bottom=run_micro, width=bar_w, color=micro_colors[i], edgecolor='black', linewidth=lw, zorder=3)
        y_text = run_micro + val
        run_micro += val

    ax2.annotate(f"${val:,.0f}", xy=(i, y_text), xytext=(0, 4), textcoords="offset points", 
                 ha='center', va='bottom', fontsize=6.5, fontweight='bold', zorder=5)

ax2.set_xticks(range(len(micro_labels)))
ax2.set_xticklabels(micro_labels, fontweight="bold") 
ax2.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)
ax2.text(0.02, 0.92, "(b)", transform=ax2.transAxes, fontweight='bold', fontsize=9)
ax2.set_ylim(top=ax2.get_ylim()[1] * 1.08)

# --- ZOOM CONNECTOR LINES ---
deg_bottom = h2 
deg_top = h2 + deg_total
bar_right_edge = 1 + (bar_w / 2) 

con_top = ConnectionPatch(xyA=(bar_right_edge, deg_top), xyB=(0, 1), 
                          coordsA="data", coordsB="axes fraction",
                          axesA=ax1, axesB=ax2, color="gray", linestyle="--", linewidth=0.5, zorder=15)

con_bottom = ConnectionPatch(xyA=(bar_right_edge, deg_bottom), xyB=(0, 0), 
                             coordsA="data", coordsB="axes fraction",
                             axesA=ax1, axesB=ax2, color="gray", linestyle="--", linewidth=0.5, zorder=15)

ax1.add_patch(con_top)
ax1.add_patch(con_bottom)

plt.tight_layout()
fig.subplots_adjust(wspace=0.2)
plt.savefig('figures/waterfall_cost_breakdown.pdf', bbox_inches='tight')
plt.show()

In [ ]:
config.k_fc / config.S_max